# Face Mask Image Classification with CNNs and OpenCV

A TensorFlow/Keras notebook that loads images from class directories, uses OpenCV for image reading, and trains a CNN classifier. The source dataset classes include masked and unmasked variants with/without sunglasses. The notebook does not implement a live webcam detection loop.

**Data and reproducibility:** No face images or trained weights are included. Place the dataset in class subdirectories under data/ and review its license and consent requirements. Training checkpoints are written to checkpoints/. Saved notebook outputs were removed.

Install the dependencies listed in `requirements.txt` in a compatible Python environment before running the notebook. The saved outputs were cleared; training and evaluation cells have not been rerun as part of this cleanup.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import image
import os
import cv2
data_path="data"
data_dir_list = ['plain-masked', 'plain-unmasked', 'sunglasses-masked', 'sunglasses-unmasked']
img_data_list=[]
labels = []
for dataset in data_dir_list:
    img_list=os.listdir(data_path+'/'+ dataset)
    print ('Loaded the images of dataset-'+'{}\n'.format(dataset))
    for img in img_list:
        if img[-3:] == 'jpg':
            input_img=cv2.imread(data_path + '/'+ dataset + '/'+ img )
#             input_img=cv2.cvtColor(input_img, cv2.COLOR_BGR2GRAY)
            labels.append(dataset)
            input_img_resize=cv2.resize(input_img,(128, 128))
            img_data_list.append(input_img_resize)
label=np.array(labels)
img_data = np.array(img_data_list)
img_data = img_data.astype('float32')
img_data = img_data/255
img_data.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the labels
label_encoded = label_encoder.fit_transform(label)

# Get the number of classes
num_classes = len(label_encoder.classes_)

# Convert to categorical
Y = to_categorical(label_encoded, num_classes)

# Proceed with shuffling and train-test split as before

x, y = shuffle(img_data, Y, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
plt.imshow(img_data_list[10])

In [ ]:
X_train = X_train.reshape(-1, 128, 128, 3)
X_test = X_test.reshape(-1, 128, 128, 3)
X_train.shape
from tensorflow.keras.layers import Conv2D, Dropout, Activation, Multiply, Input, Add, Flatten, GlobalMaxPooling2D, Dense, Reshape, BatchNormalization, GlobalAveragePooling2D, Permute
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.models import Model
import tensorflow as tf
# Explicitly name the input layer
visible = Input(shape=(128, 128, 3))

ds_1 = Conv2D(filters=32, kernel_size=(3, 3), padding='same', kernel_regularizer=regularizers.l1_l2(l1=0.001, l2=0.003), activation='relu')(visible)
x1 = BatchNormalization()(ds_1)
ds_2 = Conv2D(filters=32, kernel_size=(3, 3), padding='same', kernel_regularizer=regularizers.l1_l2(l1=0.001, l2=0.003), activation='relu')(x1)
x2 = BatchNormalization()(ds_2)
ds_3 = Conv2D(filters=32, kernel_size=(3, 3), padding='same', kernel_regularizer=regularizers.l1_l2(l1=0.001, l2=0.003), activation='relu')(x2)
x3 = BatchNormalization()(ds_3)
# ds_4 = Conv2D(filters=128, kernel_size=(3, 3), padding='same', kernel_regularizer=regularizers.l1_l2(l1=0.001, l2=0.003), activation='relu')(x3)
# x4 = BatchNormalization()(ds_4)
# ds_5 = Conv2D(filters=512, kernel_size=(3, 3), padding='same', kernel_regularizer=regularizers.l1_l2(l1=0.001, l2=0.003), activation='relu')(ds_4)
# c = cbam_attention(x4, 8)

# Residual block 1
# x = residual_block(c, filters=32, kernel_size=(3, 3))

# Spatial Pyramid Pooling (SPP)
pool_sizes = [4, 2, 1] 
spp_layers = []
for pool_size in pool_sizes:
    spp = layers.MaxPooling2D(pool_size=(pool_size, pool_size))(x3)
    spp = layers.Flatten()(spp)
    spp_layers.append(spp)

spp = layers.Concatenate()(spp_layers)
x = layers.Dense(256, activation='relu')(spp)
x1 = layers.Dropout(0.25)(x)

# Fully Connected Layers
x10 = Dense(256, activation='relu')(x1)
# x10 = Dense(128, activation='relu')(x10)

# Output Layer
output_layer = Dense(4, activation='softmax')(x10)

# Create the final model
model = Model(inputs=visible, outputs=output_layer)
model.summary()
from tensorflow.keras.optimizers import Adam
model.compile(optimizer = Adam(learning_rate = 0.001), loss='categorical_crossentropy', metrics=['acc'])
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Define the EarlyStopping callback
early_stopping = EarlyStopping(
    patience=10,
    min_delta=0.001,
    monitor="val_acc",
    restore_best_weights=True
)

# Define the ModelCheckpoint callback to save the best weights
checkpoint_directory =  'checkpoints'
epoch= 30
checkpoint = ModelCheckpoint(
   checkpoint_directory + f'Epoch_{epoch}.h5', 
    monitor='val_acc',
    save_best_only=True,
    mode='max',
    verbose=1
)
# Train the model using the new data split
history_model = model.fit(
    X_train,
    y_train,
    epochs=epoch,
    batch_size=32,
    validation_data=(X_test, y_test),  
    callbacks=[early_stopping, checkpoint],
    verbose=1
)
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")
model.evaluate(X_train, y_train)
model.evaluate(X_test,y_test)

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have a list of 'history' objects for model1 to model5
histories = [history_model]
model_names = ['Face_Mask_Detection_Model']

# Plot training & validation accuracy values for each model
for i, history in enumerate(histories):
    plt.plot(history.history['acc'], label=f'{model_names[i]} Accuracy')
    plt.plot(history.history['val_acc'], label=f'{model_names[i]} Validation Accuracy')

plt.title('Model Accuracy vs. Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy / Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score

# Predict the labels using the trained model
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Convert y_test from one-hot encoding to class indices
y_test_classes = np.argmax(y_test, axis=1)

# Making the Confusion Matrix
cm = confusion_matrix(y_test_classes, y_pred_classes)

target_names = ['sunglasses-unmasked', 'sunglasses-masked','plain-unmasked','plain-masked']

# Plot the confusion matrix as a heatmap with numbers in numeric form
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion matrix')
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
plt.show()

# Calculate and print the classification report
class_names = ['sunglasses-unmasked', 'sunglasses-masked','plain-unmasked','plain-masked']  # Replace with your actual class names
report = classification_report(y_test_classes, y_pred_classes, target_names=class_names)
print(report)

# Calculate and print the weighted F1 score
f1 = f1_score(y_test_classes, y_pred_classes, average='weighted')
print(f'Weighted F1 Score: {f1:.4f}')